# Install GLiNER2

In [1]:
pip install gliner2

   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
   - -------------------------------------- 2.9/113.8 MB 16.7 MB/s eta 0:00:07
   -- ------------------------------------- 7.1/113.8 MB 19.8 MB/s eta 0:00:06
   -- ------------------------------------- 7.6/113.8 MB 18.0 MB/s eta 0:00:06
   --- ------------------------------------ 10.2/113.8 MB 12.5 MB/s eta 0:00:09
   ----- ---------------------------------- 16.5/113.8 MB 16.0 MB/s eta 0:00:07
   ------- -------------------------------- 19.9/113.8 MB 15.9 MB/s eta 0:00:06
   -------- ------------------------------- 24.6/113.8 MB 16.8 MB/s eta 0:00:06
   ---------- ----------------------------- 29.1/113.8 MB 17.4 MB/s eta 0:00:05
   ----------- ---------------------------- 33.6/113.8 MB 17.8 MB/s eta 0:00:05
   ------------- -------------------------- 38.5/113.8 MB 18.4 MB/s eta 0:00:05
   --------------- ------------------------ 43.3/113.8 MB 18.7 MB/s eta 0:00:04
   ---------------- ----------------------- 48.2/113


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# First test with one sentence

In [2]:
from gliner2 import GLiNER2

# Load model once, use everywhere
extractor = GLiNER2.from_pretrained("fastino/gliner2-base-v1")

# Extract entities in one line
text = "Apple CEO Tim Cook announced iPhone 15 in Cupertino yesterday."
result = extractor.extract_entities(text, ["company", "person", "product", "location"])

print(result)

config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

C:\Users\nouha.laamech\AppData\Local\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nouha.laamech\.cache\huggingface\hub\models--fastino--gliner2-base-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/823 [00:00<?, ?B/s]

You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


model.safetensors:   0%|          | 0.00/834M [00:00<?, ?B/s]

{'entities': {'company': ['Apple'], 'person': ['Tim Cook'], 'product': ['iPhone 15'], 'location': ['Cupertino']}}


# Entities extraction from a given JSON file

In [32]:
# 2) Labels
labels = {
    "subject":  "we, who performs the action",
    "action":   "verb (share/collect/use/process/choose/obtain/access)",
    "object":   "thing being acted on. data/information.object",
    "purpose":  "why (advertising/analytics/service)",
    "location": "where (EU/US/servers)",
    "time":     "when/how long (during signup/for 30 days)",
    "manner":   "how/condition (with consent/as required by law)",
    "context":   "details about the action"
}

# 3) Simple sentence split
def split_statements(text):
    return [s.strip() for s in re.split(r"(?<=[.!?;])\s+|\n+", text.strip()) if s.strip()]

# 4) Make GLiNER2 output always a list of entity dicts
def entities_list(ents):
    if isinstance(ents, dict) and "entities" in ents:
        ents = ents["entities"]
    if isinstance(ents, dict):  # dict keyed by label
        out = []
        for lab, v in ents.items():
            if isinstance(v, list):
                for it in v:
                    out.append(it if isinstance(it, dict) else {"label": lab, "text": str(it), "score": 0})
            else:
                out.append(v if isinstance(v, dict) else {"label": lab, "text": str(v), "score": 0})
        ents = out
    return ents if isinstance(ents, list) else []

# 5) Decide if a sentence is important (keep a sentence if it has an action AND at least one of (object, purpose, manner, location, time))
ACTION_WORDS = re.compile(r"\b(collect|use|process|share|disclose|transfer|sell|retain|store)\b", re.I)

def is_important(sentence, fields):
    # must have an action (either extracted or keyword)
    has_action = bool(fields.get("action")) or bool(ACTION_WORDS.search(sentence))
    if not has_action:
        return False
    # must have some payload besides action
    return any(fields.get(k) for k in ["object", "purpose", "manner", "location", "time"])

# 6) Read input
input_path = r"C:\Users\nouha.laamech\Downloads\Claim verification\input.json"
with open(input_path, "r", encoding="utf-8") as f:
    text = json.load(f)["text"]

# 7) Extract only important sentences
out = {"important": []}

for s in split_statements(text):
    ents = entities_list(extractor.extract_entities(s, labels))

    fields = {k: None for k in labels}
    for e in sorted(ents, key=lambda x: x.get("score", 0), reverse=True):
        lab = str(e.get("label", "")).lower().strip()
        if lab in fields and fields[lab] is None:
            fields[lab] = e.get("text")

    # fallback for manner ("with ...")
    if fields["manner"] is None:
        m = re.search(r"\bwith\b\s+.+$", s, re.I)
        if m:
            fields["manner"] = m.group(0).strip()

    if is_important(s, fields):
        out["important"].append({"sentence": s, "fields": fields})

# 8) Save output
output_path = r"C:\Users\nouha.laamech\Downloads\Claim verification\output.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print("Saved:", output_path, "| kept:", len(out["important"]))

Saved: C:\Users\nouha.laamech\Downloads\Claim verification\output.json | kept: 4
